# DINOv2 (ViT-L/14) Feature Extraction + Linear Probe on DiopSIS

Evaluates DINOv2 — Meta's self-supervised vision foundation model — as an alternative to BioCLIP 2 for insect order classification.

**Why compare:** DINOv2 was trained on ~142M general internet images without any biology-specific data. BioCLIP 2 was trained on ~200M biological images with taxonomic captions. This experiment tests whether biology-specific pretraining (BioCLIP 2) outperforms general visual pretraining (DINOv2) on this specific downstream task.

**Setup:**
- Model: DINOv2 ViT-L/14 (300M params, 1024-dim features), matching BioCLIP 2's vision encoder size.
- Evaluation: 3-fold stratified CV with linear probe (multinomial logistic regression), matching the BioCLIP 2 linear probe protocol exactly.

**Outputs:**
- `output_dinov2/per_fold_per_class_results.csv` — per-fold per-class metrics.
- `output_dinov2/per_class_summary_mean_std.csv` — per-class mean ± std.
- `output_dinov2/headline_metrics_mean_std.csv` — aggregate metrics.

## Imports and configuration

In [1]:
import re
import time
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import normalize as sk_normalize
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.8.0+cu128
CUDA: True
GPU: NVIDIA A100-SXM4-80GB


In [2]:
CONFIG = {
    # DINOv2 ViT-L/14 — matches BioCLIP 2's vision encoder size
    "dinov2_model": "dinov2_vitl14",
    "feature_dim": 1024,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    "meta_path": "/workspace/thesis/ultimate_bioclip_top1_rank_order_meta_12062025.csv",
    "image_root": "/workspace/thesis/ALL_copy",
    "output_dir": "/workspace/thesis/output_dinov2",

    "image_id_col": "image_id",
    "label_col": "label",

    "k_folds": 3,
    "random_state": 0,
    "min_class_size": 50,

    # Image preprocessing (DINOv2 expects 224x224, ImageNet normalization)
    "image_size": 224,
    "imagenet_mean": [0.485, 0.456, 0.406],
    "imagenet_std": [0.229, 0.224, 0.225],

    "batch_size": 64,
    "num_workers": 4,

    # Linear probe (matches BioCLIP 2 linear probe config)
    "linear_probe_C": 10.0,
    "linear_probe_max_iter": 5000,
}

CROP_ID_PATTERN = re.compile(r"(\d{14}_crop_[a-zA-Z0-9]+\.jpg)$")
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)

print(f"Model: {CONFIG['dinov2_model']}")
print(f"Feature dim: {CONFIG['feature_dim']}")
print(f"Output: {CONFIG['output_dir']}")

Model: dinov2_vitl14
Feature dim: 1024
Output: /workspace/thesis/output_dinov2


## Load DINOv2 from torch.hub

DINOv2 is loaded via `torch.hub` from Meta's official release. The model is self-supervised — no classification head, just outputs feature vectors directly.

The model outputs 1024-dim features for `vitl14`. We'll use the CLS token output (single vector per image).

First run downloads weights (~1.2 GB), subsequent runs use cached weights.

In [3]:
print(f"Loading DINOv2 {CONFIG['dinov2_model']} from torch.hub...")
model = torch.hub.load('facebookresearch/dinov2', CONFIG['dinov2_model'])
model = model.to(CONFIG["device"]).eval()
print("Done.")
print(f"Model parameter count: {sum(p.numel() for p in model.parameters()):,}")

Loading DINOv2 dinov2_vitl14 from torch.hub...
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain.pth


100%|██████████| 1.13G/1.13G [00:05<00:00, 240MB/s] 


Done.
Model parameter count: 304,368,640


## Image preprocessing pipeline

DINOv2 expects:
- 224×224 RGB images (with bicubic interpolation during resize)
- ImageNet normalization (different from BioCLIP's!)
- No special preprocessing tokens

For a fair comparison with BioCLIP 2, we use the same source crops and resize them consistently. Note that BioCLIP 2 uses different normalization constants, but each model gets its appropriate preprocessing.

In [4]:
preprocess = transforms.Compose([
    transforms.Resize(CONFIG["image_size"], interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(CONFIG["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize(mean=CONFIG["imagenet_mean"], std=CONFIG["imagenet_std"]),
])

print("Preprocessing pipeline:")
print(preprocess)

Preprocessing pipeline:
Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


## Helper functions

In [5]:
def build_path_index(image_root):
    """Scanning local image directory, building {image_id: full_path}."""
    image_root = Path(image_root)
    files = list(image_root.glob("*.jpg"))
    path_index = {}
    for f in files:
        m = CROP_ID_PATTERN.search(f.name)
        if m:
            path_index[m.group(1)] = str(f)
    print(f"Mapped {len(path_index)} image_ids")
    return path_index


class ImageOrderDataset(Dataset):
    def __init__(self, image_paths, labels, preprocess):
        self.image_paths = image_paths
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.preprocess = preprocess
    def __len__(self):
        return len(self.image_paths)
    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        img = self.preprocess(img)
        return img, self.labels[idx]


def extract_dinov2_features(model, dataset, config):
    """Extracting L2-normalized DINOv2 features (CLS token output).

    DINOv2's forward() returns the CLS token feature directly (1024-dim for ViT-L/14).
    """
    device = config["device"]
    model.eval()
    loader = DataLoader(
        dataset, batch_size=config["batch_size"], shuffle=False,
        num_workers=config["num_workers"], pin_memory=True,
    )
    all_features = []
    with torch.no_grad():
        for imgs, _ in loader:
            # DINOv2's forward returns the CLS token feature
            features = model(imgs.to(device, non_blocking=True))
            features = F.normalize(features, dim=-1)
            all_features.append(features.cpu().numpy())
    return np.concatenate(all_features, axis=0)


def fit_linear_probe(train_features, train_labels, val_features, config):
    """Fitting multinomial logistic regression."""
    clf = LogisticRegression(
        C=config["linear_probe_C"], max_iter=config["linear_probe_max_iter"],
        solver="lbfgs", random_state=config["random_state"],
    )
    clf.fit(train_features, train_labels)
    return clf.predict(val_features)

## Loading metadata and aligning with local images

In [6]:
path_index = build_path_index(CONFIG["image_root"])

meta = pd.read_csv(CONFIG["meta_path"])
meta = meta.dropna(subset=[CONFIG["label_col"], CONFIG["image_id_col"]]).reset_index(drop=True)
meta["local_path"] = meta[CONFIG["image_id_col"]].map(path_index)
matched = meta.dropna(subset=["local_path"]).reset_index(drop=True)

y_str = matched[CONFIG["label_col"]].to_numpy()
image_paths = matched["local_path"].to_numpy()
classnames = sorted(np.unique(y_str).tolist())
class_to_idx = {c: i for i, c in enumerate(classnames)}
y = np.array([class_to_idx[s] for s in y_str])

class_counts = pd.Series(y_str).value_counts()
eligible_classes = class_counts[class_counts >= CONFIG["min_class_size"]].index.tolist()
eligible_class_set = set(eligible_classes)

print(f"Matched: {len(matched)} crops")
print(f"Classes ({len(classnames)}): {classnames}")
print(f"Classes with n>=50: {eligible_classes}")

Mapped 48095 image_ids
Matched: 39787 crops
Classes (14): ['Araneae', 'Blattodea', 'Coleoptera', 'Diptera', 'Hemiptera', 'Hymenoptera', 'Ixodida', 'Lepidoptera', 'Mecoptera', 'Neuroptera', 'Orthoptera', 'Plecoptera', 'Raphidioptera', 'Trichoptera']
Classes with n>=50: ['Diptera', 'Hymenoptera', 'Lepidoptera', 'Coleoptera', 'Hemiptera', 'Araneae', 'Neuroptera', 'Trichoptera', 'Plecoptera']


## Extracting DINOv2 features for the entire dataset

We extract features once for all matched images. Then for each CV fold, we just select the relevant rows from the precomputed features and fit a linear probe.

This is much faster than recomputing features per fold, and is the standard linear probe protocol.

In [7]:
print(f"Extracting DINOv2 features for {len(image_paths)} images...")
print(f"(This is a one-time pass; CV folds will reuse these features.)")

full_dataset = ImageOrderDataset(image_paths, y, preprocess)
extraction_start = time.time()
all_features = extract_dinov2_features(model, full_dataset, CONFIG)
extraction_time = time.time() - extraction_start

print(f"\nFeatures shape: {all_features.shape}")
print(f"Extraction time: {timedelta(seconds=int(extraction_time))}")

# L2-normalize features (standard for cosine-similarity-style classifiers)
all_features = sk_normalize(all_features, norm="l2", axis=1)
print("Features L2-normalized.")

Extracting DINOv2 features for 39787 images...
(This is a one-time pass; CV folds will reuse these features.)

Features shape: (39787, 1024)
Extraction time: 0:06:13
Features L2-normalized.


## Setting up 3-fold stratified CV (handling singletons)

Same protocol as BioCLIP 2 linear probe for direct comparability. Classes with n<3 (Ixodida n=1, Raphidioptera n=2) go entirely into training every fold.

In [8]:
class_counts_arr = np.bincount(y, minlength=len(classnames))
splittable_mask = class_counts_arr[y] >= CONFIG["k_folds"]
splittable_indices = np.where(splittable_mask)[0]
unsplittable_indices = np.where(~splittable_mask)[0]

print(f"Splittable samples (n>={CONFIG['k_folds']}): {len(splittable_indices)}")
print(f"Unsplittable samples (n<{CONFIG['k_folds']}, in train every fold): {len(unsplittable_indices)}")

skf = StratifiedKFold(n_splits=CONFIG["k_folds"], shuffle=True, random_state=CONFIG["random_state"])
raw_splits = list(skf.split(splittable_indices, y[splittable_indices]))

fold_splits = []
for train_rel, val_rel in raw_splits:
    train_idx = np.concatenate([splittable_indices[train_rel], unsplittable_indices])
    val_idx = splittable_indices[val_rel]
    fold_splits.append((train_idx, val_idx))

for i, (tr, va) in enumerate(fold_splits):
    print(f"  Fold {i+1}: train={len(tr)}, val={len(va)}")

Splittable samples (n>=3): 39784
Unsplittable samples (n<3, in train every fold): 3
  Fold 1: train=26525, val=13262
  Fold 2: train=26526, val=13261
  Fold 3: train=26526, val=13261


## Running cross-validation

For each fold: select train and val features from the precomputed feature array, fit logistic regression, evaluate.

In [9]:
all_results = []
overall_start = time.time()

for fold_idx, (train_idx, val_idx) in enumerate(fold_splits):
    print(f"\n{'='*70}\nFOLD {fold_idx+1}/{CONFIG['k_folds']}\n{'='*70}")
    fold_start = time.time()

    train_features = all_features[train_idx]
    train_labels = y[train_idx]
    val_features = all_features[val_idx]
    val_labels = y[val_idx]

    print(f"  Train: {len(train_idx)} examples, Val: {len(val_idx)} examples")

    print("  Fitting linear probe...")
    probe_start = time.time()
    preds = fit_linear_probe(train_features, train_labels, val_features, CONFIG)
    print(f"  Probe fit time: {time.time() - probe_start:.1f}s")

    report = classification_report(
        val_labels, preds, labels=list(range(len(classnames))),
        target_names=classnames, output_dict=True, zero_division=0,
    )

    for cls_name in classnames:
        stats = report.get(cls_name, {})
        all_results.append({
            "method": "dinov2_linear_probe",
            "fold": fold_idx,
            "class": cls_name,
            "precision": round(stats.get("precision", 0.0), 4),
            "recall": round(stats.get("recall", 0.0), 4),
            "f1": round(stats.get("f1-score", 0.0), 4),
            "support": int(stats.get("support", 0)),
        })

    # Saving intermediate
    pd.DataFrame(all_results).to_csv(
        Path(CONFIG["output_dir"]) / "per_fold_per_class_results.csv", index=False
    )

    print(f"  Fold {fold_idx+1} time: {timedelta(seconds=int(time.time() - fold_start))}")

print(f"\n{'='*70}")
print(f"CV COMPLETE — Total: {timedelta(seconds=int(time.time() - overall_start))}")
print(f"{'='*70}")


FOLD 1/3
  Train: 26525 examples, Val: 13262 examples
  Fitting linear probe...
  Probe fit time: 13.0s
  Fold 1 time: 0:00:13

FOLD 2/3
  Train: 26526 examples, Val: 13261 examples
  Fitting linear probe...
  Probe fit time: 11.2s
  Fold 2 time: 0:00:11

FOLD 3/3
  Train: 26526 examples, Val: 13261 examples
  Fitting linear probe...
  Probe fit time: 11.8s
  Fold 3 time: 0:00:11

CV COMPLETE — Total: 0:00:36


## Additional k-NN evaluation (k=5)

In [10]:
from sklearn.neighbors import KNeighborsClassifier

print("\n=== k-NN evaluation (k=5, cosine distance) ===")
knn_results = []

for fold_idx, (train_idx, val_idx) in enumerate(fold_splits):
    train_features = all_features[train_idx]
    train_labels = y[train_idx]
    val_features = all_features[val_idx]
    val_labels = y[val_idx]

    knn = KNeighborsClassifier(n_neighbors=5, metric='cosine', n_jobs=-1)
    knn.fit(train_features, train_labels)
    preds = knn.predict(val_features)

    report = classification_report(
        val_labels, preds, labels=list(range(len(classnames))),
        target_names=classnames, output_dict=True, zero_division=0,
    )
    weighted_f1 = report["weighted avg"]["f1-score"]
    print(f"  Fold {fold_idx+1}: k-NN weighted F1 = {weighted_f1:.4f}")

    knn_results.append({
        "method": "dinov2_knn", "fold": fold_idx,
        "weighted_f1": round(weighted_f1, 4),
    })

pd.DataFrame(knn_results)


=== k-NN evaluation (k=5, cosine distance) ===
  Fold 1: k-NN weighted F1 = 0.9738
  Fold 2: k-NN weighted F1 = 0.9729
  Fold 3: k-NN weighted F1 = 0.9731


,method,fold,weighted_f1
0,dinov2_knn,0,0.9738
1,dinov2_knn,1,0.9729
2,dinov2_knn,2,0.9731


## Aggregate: per-class mean and std across folds

In [11]:
results_df = pd.DataFrame(all_results)

summary_rows = []
for cls_name in classnames:
    class_data = results_df[results_df["class"] == cls_name]
    summary_rows.append({
        "class": cls_name,
        "support_total": int(class_data["support"].sum()),
        "f1_mean": round(class_data["f1"].mean(), 4),
        "f1_std": round(class_data["f1"].std(ddof=1), 4),
        "precision_mean": round(class_data["precision"].mean(), 4),
        "precision_std": round(class_data["precision"].std(ddof=1), 4),
        "recall_mean": round(class_data["recall"].mean(), 4),
        "recall_std": round(class_data["recall"].std(ddof=1), 4),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(Path(CONFIG["output_dir"]) / "per_class_summary_mean_std.csv", index=False)
summary_df

,class,support_total,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std
0,Araneae,564,0.9929,0.0016,0.9965,0.0061,0.9894,0.0054
1,Blattodea,23,0.9361,0.0625,0.9583,0.0722,0.9167,0.0722
2,Coleoptera,1143,0.8634,0.0170,0.9125,0.0109,0.8198,0.0324
3,Diptera,31913,0.9881,0.0010,0.9827,0.0006,0.9937,0.0016
4,Hemiptera,775,0.8430,0.0070,0.9128,0.0050,0.7832,0.0144
5,Hymenoptera,3326,0.9391,0.0019,0.9463,0.0064,0.9321,0.0051
6,Ixodida,0,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
7,Lepidoptera,1744,0.9853,0.0030,0.9885,0.0039,0.9823,0.0052
8,Mecoptera,34,0.1545,0.0119,0.6667,0.2887,0.0884,0.0044
9,Neuroptera,115,0.6804,0.0671,0.8013,0.0824,0.5918,0.0607


## Headline aggregate metrics (mean ± std)

In [12]:
per_fold_weighted = []
per_fold_macro_all = []
per_fold_macro_eligible = []
per_fold_hemi = []
per_fold_cole = []

for fold in range(CONFIG["k_folds"]):
    fold_data = results_df[results_df["fold"] == fold]
    per_fold_weighted.append(np.average(fold_data["f1"], weights=fold_data["support"]))
    per_fold_macro_all.append(fold_data["f1"].mean())
    eligible_data = fold_data[fold_data["class"].isin(eligible_class_set)]
    per_fold_macro_eligible.append(eligible_data["f1"].mean())
    per_fold_hemi.append(fold_data[fold_data["class"] == "Hemiptera"]["f1"].iloc[0])
    per_fold_cole.append(fold_data[fold_data["class"] == "Coleoptera"]["f1"].iloc[0])

headline_df = pd.DataFrame([{
    "method": "dinov2_linear_probe",
    "weighted_f1_mean": round(np.mean(per_fold_weighted), 4),
    "weighted_f1_std": round(np.std(per_fold_weighted, ddof=1), 4),
    "macro_f1_all_mean": round(np.mean(per_fold_macro_all), 4),
    "macro_f1_all_std": round(np.std(per_fold_macro_all, ddof=1), 4),
    "macro_f1_eligible_mean": round(np.mean(per_fold_macro_eligible), 4),
    "macro_f1_eligible_std": round(np.std(per_fold_macro_eligible, ddof=1), 4),
    "Hemiptera_f1_mean": round(np.mean(per_fold_hemi), 4),
    "Hemiptera_f1_std": round(np.std(per_fold_hemi, ddof=1), 4),
    "Coleoptera_f1_mean": round(np.mean(per_fold_cole), 4),
    "Coleoptera_f1_std": round(np.std(per_fold_cole, ddof=1), 4),
}])
headline_df.to_csv(Path(CONFIG["output_dir"]) / "headline_metrics_mean_std.csv", index=False)
headline_df

,method,weighted_f1_mean,weighted_f1_std,macro_f1_all_mean,macro_f1_all_std,macro_f1_eligible_mean,macro_f1_eligible_std,Hemiptera_f1_mean,Hemiptera_f1_std,Coleoptera_f1_mean,Coleoptera_f1_std
0,dinov2_linear_probe,0.9756,0.0011,0.7286,0.0026,0.9011,0.0077,0.843,0.007,0.8634,0.017


## Comparison context

The headline metrics above can be directly compared with BioCLIP 2 linear probe results from `02_cv_lora.ipynb`:

- **BioCLIP 2 + linear probe** (frozen baseline in `02_cv_lora.ipynb`): represents biology-specific pretraining + simple classifier.
- **DINOv2 + linear probe** (this notebook): represents general visual pretraining + simple classifier.

The comparison tells us whether biology-specific pretraining provides a measurable advantage on this insect classification task, beyond what general visual features can support.

## Output files summary

The notebook generates:

| File | Purpose |
|---|---|
| `per_fold_per_class_results.csv` | Granular per-fold per-class precision/recall/F1 |
| `per_class_summary_mean_std.csv` | Per-class aggregates with mean ± std across folds |
| `headline_metrics_mean_std.csv` | Top-line aggregates (weighted F1, macro F1, etc.) with mean ± std |

Comparing the headline metrics with `02_cv_lora.ipynb` (frozen BioCLIP 2 linear probe baseline) one determines whether biology-specific pretraining (BioCLIP 2) provides a measurable advantage over general self-supervised visual pretraining (DINOv2) on this insect classification task.
